# Italian Solvency reports Table S.02.01.02; Part 2 Processing
This script is a continuation of `Extraction_Italy_SII`. It shows the second Phase of this process.

## Companies in scope

For the year 2024, the companies in scope are the following:

 - Credemvita S.p.A.
 - AXA MPS Assicurazioni Vita
 - CRÈDIT AGRICOLE VITA
 - Società Reale Mutua di Assicurazioni
 - Cardif Vita S.p.A.
 - MEDIOLANUM VITA S.p.A.
 - Generali Italia S.p.A.
 - Banco BPM Vita S.p.A.
 - HDI ASSICURAZIONI S.p.A.
 - Gruppo Assicurativo Poste Vita
 - FIDEURAM VITA S.P.A.
 - CNP Vita Assicura S.p.A.
 - ITAS VITA
 - Helvetia Vita S.p.A.
 - Vittoria Assicurazioni S.p.A.
 - GROUPAMA ASSICURAZIONI S.P.A.
 - UniCredit Allianz Vita S.p.A.
 - Zurich Investments Life S.p.A.


## Description of the process

The process of extraction is performed in 5 phases:

### Phase 1: Find the reports and identify the relevant tables. 
 1) Identify the new SFCR report and save it into the folder Input.
 2) Identify the pages where the tables of interest are.
 3) Compile the map of the company run in the master_list.csv.

### Phase 2: Run the Extraction script (this script). 
The script performs the following steps (with slight modifications depending on the table format):
 1) Save the page with the table into a separate folder Single_pdf.
 2) Use either a Python package or specialized LLM to create a digital equivalent of the table.
 3) Fix the systemic errors that prevent the table from being saved as DataFrame.
 4) Save the DataFrame into the Output folder.

### Phase 3: Run the Processing script. 
The script applies fixes to the DataFrame to make the numbers closer to the reported numbers. It joins all the tables into a single dataset. 

### Phase 4: Run the Cross-Validation script. 
Applies a series of tests that check for the internal consistency between the numbers. Flags the potential errors.

### Phase 5: Final modifications to the table and a manual inspection. 

## Python packages

In [929]:
import pandas as pd
import numpy as np

## Functions

In [930]:
def remove_trailing_zeros(data: pd.DataFrame, col: str, n: int) -> pd.DataFrame:
    """
    Removes exactly n trailing zeros from numbers in a column, if present.

    Args:
        data (pd.DataFrame): Input dataframe.
        col (str): Column name to process.
        n (int): Number of trailing zeros to remove.

    Returns:
        pd.DataFrame: DataFrame with modified column.
    """
    def clean_value(x):
        if pd.isna(x):
            return x
        try:
            # Convert to int first (in case it's float or string with decimals)
            x_int = int(float(x))
            str_x = str(x_int)
            zeros = "0" * n
            if str_x.endswith(zeros):
                return int(str_x[:-n])  # remove trailing zeros
            return x_int
        except ValueError:
            return x  # return unchanged if not a valid number

    data[col] = data[col].apply(clean_value)
    return data

In [931]:
def adjust_decimals(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Adjust values in a dataframe column:
    - If a value has non-zero decimal digits, multiply it by 1000.
    - Otherwise, leave it unchanged.

    Parameters
    ----------
    data : pd.DataFrame
        Input dataframe with numeric values.
    col : str
        Column name to check (default "C0010").

    Returns
    -------
    pd.DataFrame
        Updated dataframe with adjusted values.
    """
    data = data.copy()
    decimals = data[col] % 1
    data[col] = np.where(decimals != 0, data[col] * 1000, data[col])
    return data

In [932]:
def remove_space(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Remove every space in a column of strings in a DataFrame. 
    """  
    data[col] = (
        data[col]
        .str.replace(" ", "", regex=False)   # remove thousand separators
    )
    return data

In [933]:
def remove_dot_sep(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Remove every dot in a column of strings in a DataFrame. 
    """  
    data[col] = (
        data[col]
        .str.replace(".", "", regex=False)
    )
    return data

In [934]:
def as_float(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Convert a column of strings in a DataFrame into floating numbers. 
    """  
    data[col] = (
        data[col]
        .astype(float)
    )
    return data

In [935]:
def remove_comma_sep(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Remove every comma in a column of strings in a DataFrame. 
    """  
    data[col] = (
        data[col]
        .str.replace(",", "", regex=False)
    )
    return data

In [936]:
def remove_dolar_sep(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Remove every dolar sign in a column of strings in a DataFrame. 
    """  
    data[col] = (
        data[col]
        .str.replace("$", "", regex=False)
    )
    return data

In [937]:
def replace_coma_with_dot(data: pd.DataFrame, col: str = "C0010") -> pd.DataFrame:
    """
    Replace every comma in a column of strings of a DataFrame with a dot. 
    """
    data[col] = (
        data[col]
        .str.replace(",", ".", regex=False)
    )
     
    return data

In [938]:
def fix_dot_numbers(data: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Fix numbers in a column where dots are used as thousand separators 
    but keep values like '0.000' unchanged.

    Args:
        data (pd.DataFrame): Input dataframe.
        col (str): Column name to fix.

    Returns:
        pd.DataFrame: DataFrame with the cleaned column.
    """

    def clean_value(x):
        if pd.isna(x):  # Handle NaN
            return x
        x_str = str(x)
        before, _, after = x_str.partition(".")
        if before == "0" and after == "000":
            return x_str   # keep '0.000'
        return x_str.replace(".", "")  # remove dots otherwise

    data[col] = data[col].apply(clean_value)
    return data

In [939]:
def replace_dash_with_zero(data: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Replace cells containing only '-' with 0 in the specified column.
    
    Parameters
    ----------
    data : pd.DataFrame
        Input dataframe.
    col : str
        Column name where replacement should happen.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with '-' replaced by 0 in the specified column.
    """
    data = data.copy()
    data[col] = data[col].apply(lambda x: 0 if str(x).strip() == "-" else x)
    data[col] = data[col].apply(lambda x: 0 if str(x).strip() == "–" else x)
    return data

In [940]:
def remove_bracket_instead_negative(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    Converts numbers stored as strings with brackets into negative numbers.
    Example: "(123)" -> "-123"
    
    Parameters:
        data (pd.DataFrame): Input DataFrame
        column (str): Column name where conversion should be applied
    
    Returns:
        pd.DataFrame: Updated DataFrame with cleaned column
    """
    data = data.copy()
    
    def convert(value):
        if isinstance(value, str) and value.startswith("(") and value.endswith(")"):
            return "-" + value[1:-1]  # remove brackets, prepend "-"
        return value
    
    data[column] = data[column].apply(convert)
    return data

In [941]:
def initial_read(unique_id: str, list:pd.DataFrame) -> pd.DataFrame:
    """
    Find the location of the table and load it.
    """
    path = list.loc[unique_id, "output_final_path"]

    return pd.read_csv(path, index_col = 0).fillna(0)

In [942]:
def shorten_index(data: pd.DataFrame) -> pd.DataFrame:
    """
    Trims the index values so they contain only the first 5 characters.
    
    Example:
        'R0510 2.337.991' -> 'R0510'
    """
    data = data.copy()
    data.index = data.index.astype(str).str[:5]
    return data

In [943]:
def convert_index_to_R(data: pd.DataFrame) -> pd.DataFrame:
    """
    Converts the dataframe index by replacing the first digit with 'R'
    and formatting the rest as string without decimals.
    
    Example:
        80000.0 -> "R0000"
        80500.0 -> "R0500"
    """
    data = data.copy()
    data.index = (
        data.index.astype(int).astype(str).str[1:].str.zfill(4).map(lambda x: "R" + x)
    )
    return data

# Master list of companies

In [944]:
master_list = pd.read_csv("master_list.csv", header=0, index_col=0)

In [945]:
display(master_list)

,company,document_name,table_name,page_number,output_pdf_path,output_final_path,codes_path,leto
id,,,,,,,,
CREDEM_VITA_02_1,CREDEM,Input\SFCR 2024 CREDEMVITA.pdf,S.02.01.02_A,197,Single_pdf/CREDEM_S02_01_02_1_2024.pdf,Output/CREDEM_S02_01_02_1_2024.csv,NaN,2024
CREDEM_VITA_02_2,CREDEM,Input\SFCR 2024 CREDEMVITA.pdf,S.02.01.02_L,198,Single_pdf/CREDEM_S02_01_02_2_2024.pdf,Output/CREDEM_S02_01_02_2_2024.csv,NaN,2024
AXA_VITA_02_01,AXA,Input\2024.12 QRT SFCR AXA MPS Assicurazioni V...,S.02.01.02_A,1,Single_pdf/AXA_S02_01_02_1_2024.pdf,Output/AXA_S02_01_02_1_2024.csv,NaN,2024
CREDAG_VITA_02_01,CREDIT_AGRICOLE,Input\ca_vita_sfcr_2024.pdf,S.02.01.02_A,61,Single_pdf/CA_S02_01_02_1_2024.pdf,Output/CA_S02_01_02_1_2024.csv,NaN,2024
CREDAG_VITA_02_02,CREDIT_AGRICOLE,Input\ca_vita_sfcr_2024.pdf,S.02.01.02_L,62,Single_pdf/CA_S02_01_02_2_2024.pdf,Output/CA_S02_01_02_2_2024.csv,NaN,2024
REALE_02_01,REALE_MUTUA,Input\SFCR_A123S_20241231.pdf,S.02.01.02_A,158,Single_pdf/RM_S02_01_02_1_2024.pdf,Output/RM_S02_01_02_1_2024.csv,NaN,2024
REALE_02_02,REALE_MUTUA,Input\SFCR_A123S_20241231.pdf,S.02.01.02_L,159,Single_pdf/RM_S02_01_02_2_2024.pdf,Output/RM_S02_01_02_2_2024.csv,NaN,2024
CARDIF_02_01,CARDIF,Input\SFCR_A421S_20241231.pdf,S.02.01.02_A,106,Single_pdf/CARDIF_S02_01_02_1_2024.pdf,Output/CARDIF_S02_01_02_1_2024.csv,NaN,2024
MEDIO_02_01,MEDIOLANUM,Input\Mediolanum_Vita_Relazione_Unica_2024.pdf,S.02.01.02_A,164,Single_pdf/MEDIO_S02_01_02_1_2024.pdf,Output/MEDIO_S02_01_02_1_2024.csv,NaN,2024


# Code

## Credemvita S.p.A.

S.01.02.01 1

In [946]:
table_1 = initial_read("CREDEM_VITA_02_1", master_list)

In [947]:
table_1 = remove_space(table_1,"C0010")
table_1 = remove_dot_sep(table_1,"C0010")
table_1 = as_float(table_1,"C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [948]:
table_2 = initial_read("CREDEM_VITA_02_2", master_list)

In [949]:
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [950]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["CREDEM_VITA"],["NAME","C0010"]])

In [951]:
credem_02 = table

In [952]:
del table, table_1, table_2

## AXA MPS Assicurazioni Vita

S.02.01.02

In [953]:
table = initial_read("AXA_VITA_02_01", master_list)

In [954]:
table = replace_dash_with_zero(table, "C0010")
table = remove_dot_sep(table, "C0010")
table = replace_coma_with_dot(table, "C0010")    
table = remove_dolar_sep(table, "C0010")
table = remove_space(table, "C0010")
table = as_float(table, "C0010")
table = table.fillna(0)

In [955]:
table.columns = pd.MultiIndex.from_product([["AXA"],["C0010"]])

In [956]:
axa_02 = table

In [957]:
del table

## CRÈDIT AGRICOLE VITA

S.02.01.02 1

In [958]:
table_1 = initial_read("CREDAG_VITA_02_01", master_list)

In [959]:
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = replace_coma_with_dot(table_1, "C0010")    
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [960]:
table_2 = initial_read("CREDAG_VITA_02_02", master_list)

In [961]:
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = replace_coma_with_dot(table_2, "C0010")    
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [962]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["CREDIT_AGRICOLE"],["NAME","C0010"]])

In [963]:
ca_02 = table

In [964]:
del table, table_1, table_2

## Società Reale Mutua di Assicurazioni

S.02.01.02 1

In [965]:
table_1 = initial_read("REALE_02_01", master_list)

S.02.01.02 2

In [966]:
table_2 =initial_read("REALE_02_02", master_list)

In [967]:
table_2 = shorten_index(table_2)

table_2 = remove_dot_sep(table_2, "C0010")
table_2 = replace_coma_with_dot(table_2, "C0010")    
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [968]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["REALE_MUTUA"],["NAME","C0010"]])

In [969]:
rm_02 = table

In [970]:
del table, table_1, table_2

## Cardif Vita S.p.A.

S.02.01.02

In [971]:
table = initial_read("CARDIF_02_01", master_list)

In [972]:
table = table[~table.index.isna()]

In [973]:
table = replace_dash_with_zero(table, "C0010")
table = remove_dot_sep(table, "C0010")
table = replace_coma_with_dot(table, "C0010")    
table = remove_dolar_sep(table, "C0010")
table = remove_space(table, "C0010")
table = as_float(table, "C0010")
table = table.fillna(0)

In [974]:
table.columns = pd.MultiIndex.from_product([["CARDIF"],["NAME","C0010"]])

In [975]:
cardif_02 = table

In [976]:
del table

## MEDIOLANUM VITA S.p.A.

S.02.01.02 1

In [977]:
table_1 = initial_read("MEDIO_02_01", master_list)

In [978]:
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [979]:
table_2 = initial_read("MEDIO_02_02", master_list)

In [980]:
table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [981]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["MEDIOLANUM"],["NAME","C0010"]])

In [982]:
medio_02 = table

In [983]:
del table, table_1, table_2

## Generali Italia S.p.A.

S.02.01.02 1

In [984]:
table_1 = initial_read("GEN_ITA_02_01", master_list)

In [985]:
table_1 = table_1.loc[table_1.index.notna()]

table_1 = remove_dot_sep(table_1, "C0010")
table_1 = replace_coma_with_dot(table_1, "C0010")    
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [986]:
table_2 = initial_read("GEN_ITA_02_02", master_list)

In [987]:
table_2 = table_2.loc[table_2.index.notna()]

table_2 = remove_dot_sep(table_2, "C0010")
table_2 = replace_coma_with_dot(table_2, "C0010")    
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [988]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["GENERALI_ITALIA"],["NAME","C0010"]])

In [989]:
gen_ita_02 = table

In [990]:
del table, table_1, table_2

## Banco BPM Vita S.p.A.

S.02.01.02 1

In [991]:
table_1 = initial_read("BMP_VITA_02_01", master_list)

In [992]:
table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_comma_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2 

In [993]:
table_2 = initial_read("BMP_VITA_02_02", master_list)

In [994]:
table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_comma_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [995]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["BMP_VITA"],["NAME","C0010"]])

In [996]:
bmp_vita_02 = table

In [997]:
del table, table_1, table_2

## HDI ASSICURAZIONI S.p.A.

In [998]:
table_1 = initial_read("HDI_01", master_list)

In [999]:
table_1 = remove_space(table_1, "C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1000]:
table_2 = initial_read("HDI_02", master_list)

In [1001]:
table_2 = table_2[~table_2.index.isna()]
table_2 = table_2[~(table_2.index == '---')]
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1002]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["HDI"],["NAME","C0010"]])

In [1003]:
 hdi_02 = table

In [1004]:
del table, table_1, table_2

## Gruppo Assicurativo Poste Vita

S.02.01.02 1

In [1005]:
table_1 = initial_read("POSTE_VITA_01", master_list)

In [1006]:
table_1 = remove_space(table_1, "C0010")
table_1 = replace_dash_with_zero(table_1,"C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_bracket_instead_negative(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1007]:
table_2 = initial_read("POSTE_VITA_02", master_list)

In [1008]:
table_2 = table_2.loc[table_2.index!="-"]

table_2 = remove_space(table_2, "C0010")
table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_bracket_instead_negative(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1009]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["POSTE_VITA"],["NAME","C0010"]])

In [1010]:
poste_vita_02 = table

In [1011]:
del table, table_1, table_2

## FIDEURAM VITA S.P.A.

S.02.01.02 1

In [1012]:
table_1 = initial_read("INTESA_VITA_01", master_list)

In [1013]:
table_1 = table_1.loc[table_1.index.notna()]

table_1 = replace_dash_with_zero(table_1,"C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_comma_sep(table_1,"C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1014]:
table_2 = initial_read("INTESA_VITA_02", master_list)

In [1015]:
table_2 = table_2.loc[table_2.index.notna()]

table_2 = replace_dash_with_zero(table_2,"C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_comma_sep(table_2,"C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1016]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["INTESA_VITA"],["NAME","C0010"]])

In [1017]:
intesa_vita_02 = table

In [1018]:
del table, table_1, table_2

## CNP Vita Assicura S.p.A.

S.02.01.02 1

In [1019]:
table_1 = initial_read("CNP_VITA_01", master_list)

In [1020]:
table_1 = table_1[~table_1.index.isna()]
table_1 = convert_index_to_R(table_1)

table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1021]:
table_2 = initial_read("CNP_VITA_02", master_list)

In [1022]:
table_2 = convert_index_to_R(table_2)

table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1023]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["CNP_VITA"],["NAME","C0010"]])

In [1024]:
cnp_vita_02 = table

In [1025]:
del table, table_1, table_2

## ITAS VITA

S.02.01.02 1

In [1026]:
table_1 = initial_read("ITAS_VITA_01", master_list)

In [1027]:
table_1 = table_1.loc[table_1.index.notna()]

table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2 

In [1028]:
table_2 = initial_read("ITAS_VITA_02", master_list)

In [1029]:
table_2 = table_2[~table_2.index.isna()]

table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1030]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["ITAS_VITA"],["NAME","C0010"]])

In [1031]:
itas_vita_02 = table

In [1032]:
del table, table_1, table_2

## Helvetia Vita S.p.A.

S.02.01.02 1

In [1033]:
table_1 = initial_read("HELVETIA_VITA_01", master_list)

In [1034]:
table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1035]:
table_2 = initial_read("HELVETIA_VITA_02", master_list)

In [1036]:
table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_dot_sep(table_2,"C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1037]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["HELVATIA_VITA"],["NAME","C0010"]])

In [1038]:
helvatia_vita_02 = table

In [1039]:
del table, table_1, table_2

## Vittoria Assicurazioni S.p.A.

S.02.01.02 1

In [1040]:
table_1 = initial_read("VITTORIA_01", master_list)

In [1041]:
table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_bracket_instead_negative(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1042]:
table_2 = initial_read("VITTORIA_02", master_list)

In [1043]:
table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")
table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_bracket_instead_negative(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1044]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["VITTORIA"],["C0010"]])

In [1045]:
vittora_vita_02 = table

In [1046]:
del table, table_1, table_2

## GROUPAMA ASSICURAZIONI S.P.A.

S.02.01.02 1

In [1047]:
table = initial_read("GROUPAMA_01", master_list)

In [1048]:
table = table[~table.index.isna()]
table = table[~(table.index == '---')]

In [1049]:
table = replace_dash_with_zero(table,"C0010")
table = remove_space(table, "C0010")
table = remove_dolar_sep(table, "C0010")
table = remove_bracket_instead_negative(table, "C0010")
table = remove_dot_sep(table,"C0010")
table = as_float(table, "C0010")
table = table.fillna(0)

In [1050]:
table.columns = pd.MultiIndex.from_product([["GROUPAMA"],["NAME","C0010"]])

In [1051]:
groupama_02 = table

In [1052]:
del table

## UniCredit Allianz Vita S.p.A.

S.02.01.02 1 

In [1053]:
table_1 = initial_read("ALLIANZ_UNICREDIT_01", master_list)

In [1054]:
table_1 = table_1.loc[table_1.index.notna()]

table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")

table_1 = remove_dolar_sep(table_1, "C0010")
table_1 = remove_bracket_instead_negative(table_1, "C0010")
table_1 = remove_dot_sep(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1055]:
table_2 = initial_read("ALLIANZ_UNICREDIT_02", master_list)

In [1056]:
table_2 = table_2.loc[table_2.index.notna()]

table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = remove_space(table_2, "C0010")

table_2 = remove_dolar_sep(table_2, "C0010")
table_2 = remove_bracket_instead_negative(table_2, "C0010")
table_2 = remove_dot_sep(table_2, "C0010")
table_2 = as_float(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1057]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["ALLIANZ_UNICREDIT"],["NAME","C0010"]])

In [1058]:
allianz_02 = table

In [1059]:
del table, table_1, table_2

## Zurich Investments Life S.p.A.

In [1060]:
table_1 = initial_read("ZURICH_LIFE_01", master_list)

In [1061]:
table_1 = replace_dash_with_zero(table_1, "C0010")
table_1 = remove_space(table_1, "C0010")
table_1 = as_float(table_1, "C0010")
table_1 = table_1.fillna(0)

S.02.01.02 2

In [1062]:
table_2 = initial_read("ZURICH_LIFE_02", master_list)

In [1063]:
table_2 = replace_dash_with_zero(table_2, "C0010")
table_2 = table_2.fillna(0)

In [1064]:
table = pd.concat([table_1, table_2])
table.columns = pd.MultiIndex.from_product([["ZURICH_LIFE"],["NAME","C0010"]])

In [1065]:
zurich_life_02 = table

In [1066]:
del table, table_1, table_2

## Dirty master table

In [1067]:
master_ita_02 = pd.read_csv("Input/S02_01_02_Master_Table.csv", index_col = 0).fillna(0) 
master_ita_02.columns = pd.MultiIndex.from_product([["MASTER"],["NAME"]])

In [1068]:
credem_02 = credem_02.drop(columns=("CREDEM_VITA","NAME"))
#axa_02 = axa_02.drop(columns=("AXA","NAME"))
ca_02 = ca_02.drop(columns=("CREDIT_AGRICOLE","NAME"))
rm_02 = rm_02.drop(columns=("REALE_MUTUA","NAME"))
cardif_02 = cardif_02.drop(columns=("CARDIF","NAME"))
medio_02 = medio_02.drop(columns=("MEDIOLANUM","NAME"))
gen_ita_02 = gen_ita_02.drop(columns=("GENERALI_ITALIA","NAME"))
bmp_vita_02 = bmp_vita_02.drop(columns=("BMP_VITA","NAME"))
hdi_02 = hdi_02.drop(columns=("HDI","NAME"))
poste_vita_02 = poste_vita_02.drop(columns=("POSTE_VITA","NAME"))
intesa_vita_02 = intesa_vita_02.drop(columns=("INTESA_VITA","NAME"))
cnp_vita_02 = cnp_vita_02.drop(columns=("CNP_VITA","NAME"))
itas_vita_02 = itas_vita_02.drop(columns=("ITAS_VITA","NAME"))
helvatia_vita_02 = helvatia_vita_02.drop(columns=("HELVATIA_VITA","NAME"))
#vittora_vita_02 = vittora_vita_02.drop(columns=("VITTORIA","NAME"))
groupama_02 = groupama_02.drop(columns=("GROUPAMA","NAME"))
allianz_02 = allianz_02.drop(columns=("ALLIANZ_UNICREDIT","NAME"))
zurich_life_02 = zurich_life_02.drop(columns=("ZURICH_LIFE","NAME"))

In [1069]:
table_02 = master_ita_02.join(credem_02).join(axa_02).join(ca_02).join(rm_02).join(cardif_02).join(medio_02).join(gen_ita_02).join(bmp_vita_02).join(hdi_02).join(poste_vita_02).join(intesa_vita_02).join(cnp_vita_02).join(itas_vita_02).join(helvatia_vita_02).join(vittora_vita_02).join(groupama_02).join(allianz_02).join(zurich_life_02)

In [1070]:
table_02 = table_02[~table_02.index.duplicated(keep=False)]

In [1071]:
table_02.to_csv("Dirty_Combined/Table_ita_S02.csv")